In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

DATABASE = "KMAT_COST_MODEL_DB"
SCHEMA = "CORE_ML"
SOURCE = f"{DATABASE}.{SCHEMA}.TDS_MODEL_DATASET_V2"
TARGET = "TARGET_TDS_FACTOR"
RULE = "DEVELOPMENT_RULE_BASELINE_FACTOR"
LOWER, UPPER = 1.0, 3.5
HIGH_STRAIN = 2.0
SEVERE_GAP = 0.25
RANDOM_STATE = 42

CAT_COLS = [
    "KMAT_ID", "WORK_CENTER_ID", "MACHINE_ID", "OPERATION_ID",
    "ENGINE", "CAB", "WHEEL", "COLOR",
]
NUM_COLS = [
    "MACHINE_AGE_YEARS", "MACHINE_HOURS_RUN", "PRE_RUN_VIBRATION_MM_S",
    "PRE_RUN_TEMPERATURE_C", "DAYS_SINCE_MAINTENANCE",
    "TOOL_USAGE_CYCLES_PRE_RUN", "SETUP_TIME_MINUTES",
    "CONFIG_COMPLEXITY_SCORE", "STANDARD_MAINTENANCE_COST_USD",
    "SHIFT_LOAD_PCT", "OPERATOR_EXPERIENCE_YEARS",
    "RECENT_30D_DOWNTIME_MIN", "PRIOR_30D_MAINTENANCE_COUNT",
    "MACHINE_HEALTH_SCORE", "PRIOR_30D_MEAN_VIBRATION_MM_S",
    "PRIOR_WORK_CENTER_TDS_AVG",
]
FEATURES = CAT_COLS + NUM_COLS


def utc_now():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def clip(values):
    return np.clip(np.asarray(values, dtype=float), LOWER, UPPER)


def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = clip(predicted)
    high = actual >= HIGH_STRAIN
    under = predicted < actual
    severe = (actual - predicted) >= SEVERE_GAP
    return {
        "ROW_COUNT": int(len(actual)),
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(mean_squared_error(actual, predicted) ** 0.5),
        "MEDIAN_ABSOLUTE_ERROR": float(median_absolute_error(actual, predicted)),
        "UNDERPREDICTION_RATE": float(under.mean()),
        "SEVERE_UNDERPREDICTION_RATE": float(severe.mean()),
        "HIGH_STRAIN_ROW_COUNT": int(high.sum()),
        "HIGH_STRAIN_MAE": float(mean_absolute_error(actual[high], predicted[high])) if high.any() else None,
        "HIGH_STRAIN_UNDERPREDICTION_RATE": float((predicted[high] < actual[high]).mean()) if high.any() else None,
        "HIGH_STRAIN_SEVERE_UNDERPREDICTION_RATE": float(((actual[high] - predicted[high]) >= SEVERE_GAP).mean()) if high.any() else None,
        "PREDICTION_MIN": float(predicted.min()),
        "PREDICTION_MAX": float(predicted.max()),
        "RANGE_VIOLATION_ROWS": int(((predicted < LOWER) | (predicted > UPPER)).sum()),
        "MEAN_ERROR": float((predicted - actual).mean()),
    }


def preprocessor(scale_numeric=False):
    return ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
            ("num", StandardScaler() if scale_numeric else "passthrough", NUM_COLS),
        ],
        remainder="drop",
    )


def candidates():
    return {
        "LINEAR_REGRESSION": Pipeline([
            ("prep", preprocessor(True)),
            ("model", LinearRegression()),
        ]),
        "RANDOM_FOREST_V1": Pipeline([
            ("prep", preprocessor(False)),
            ("model", RandomForestRegressor(
                n_estimators=250, max_depth=14, min_samples_leaf=5,
                max_features=0.8, random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ]),
        "XGB_V1_BALANCED": Pipeline([
            ("prep", preprocessor(False)),
            ("model", XGBRegressor(
                objective="reg:squarederror", n_estimators=350,
                learning_rate=0.035, max_depth=5, min_child_weight=5,
                subsample=0.85, colsample_bytree=0.85,
                reg_alpha=0.05, reg_lambda=1.25,
                random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ]),
        "XGB_V2_CONSERVATIVE": Pipeline([
            ("prep", preprocessor(False)),
            ("model", XGBRegressor(
                objective="reg:squarederror", n_estimators=450,
                learning_rate=0.025, max_depth=4, min_child_weight=8,
                subsample=0.90, colsample_bytree=0.90,
                reg_alpha=0.10, reg_lambda=1.75,
                random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ]),
        "XGB_V3_HIGH_STRAIN": Pipeline([
            ("prep", preprocessor(False)),
            ("model", XGBRegressor(
                objective="reg:pseudohubererror", n_estimators=400,
                learning_rate=0.03, max_depth=6, min_child_weight=4,
                subsample=0.85, colsample_bytree=0.90,
                reg_alpha=0.05, reg_lambda=1.50,
                random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ]),
    }


def save(session, pdf, name):
    session.write_pandas(
        pdf, table_name=name, database=DATABASE, schema=SCHEMA,
        auto_create_table=True, overwrite=True, quote_identifiers=False,
    )


def validate(df):
    required = {"ENTITY_ID", "FEATURE_AS_OF_DATE", "DATA_SPLIT", TARGET, RULE, *FEATURES}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if df.empty or df["ENTITY_ID"].isna().any() or df["ENTITY_ID"].duplicated().any():
        raise ValueError("Invalid or duplicate ENTITY_ID values.")
    if df[TARGET].isna().any() or not df[TARGET].between(LOWER, UPPER).all():
        raise ValueError("Invalid TDS target values.")
    if df[FEATURES].isna().any().any():
        raise ValueError("Null model features found.")
    if set(df["DATA_SPLIT"].unique()) != {"TRAIN", "VALIDATION", "TEST"}:
        raise ValueError("TRAIN, VALIDATION and TEST splits are required.")


def prediction_frame(source, predicted, model_name, split):
    result = source[[
        "ENTITY_ID", "FEATURE_AS_OF_DATE", "DATA_SPLIT", "WORK_CENTER_ID",
        "MACHINE_ID", "OPERATION_ID", "ENGINE", "CAB", "WHEEL", TARGET, RULE,
    ]].copy()
    raw = np.asarray(predicted, dtype=float)
    pred = clip(raw)
    actual = result[TARGET].to_numpy(dtype=float)
    result["RAW_PREDICTED_TDS_FACTOR"] = raw
    result["PREDICTED_TDS_FACTOR"] = pred
    result["ABSOLUTE_ERROR"] = np.abs(pred - actual)
    result["UNDERPREDICTION_FLAG"] = (pred < actual).astype(int)
    result["SEVERE_UNDERPREDICTION_FLAG"] = ((actual - pred) >= SEVERE_GAP).astype(int)
    result["HIGH_STRAIN_FLAG"] = (actual >= HIGH_STRAIN).astype(int)
    result["MODEL_NAME"] = model_name
    result["MODEL_VERSION"] = "V2"
    result["FEATURE_SET_VERSION"] = "TDS_FEATURES_V2"
    result["EVALUATION_SPLIT"] = split
    result["DEPLOYMENT_MODE"] = "SHADOW"
    result["OFFICIAL_COST_IMPACT_ALLOWED_FLAG"] = False
    result["BUSINESS_DECISION_ALLOWED_FLAG"] = False
    result["PREDICTED_AT"] = utc_now()
    return result


def run(session: Session):
    df = session.table(SOURCE).to_pandas()
    df.columns = [str(c).upper() for c in df.columns]
    validate(df)

    train = df[df.DATA_SPLIT == "TRAIN"].copy()
    val = df[df.DATA_SPLIT == "VALIDATION"].copy()
    test = df[df.DATA_SPLIT == "TEST"].copy()

    rows = []
    rule_val = metrics(val[TARGET], val[RULE])
    rows.append({"MODEL_NAME": "DEVELOPMENT_RULE_BASELINE", "MODEL_TYPE": "RULE", **rule_val})
    median_val = metrics(val[TARGET], np.full(len(val), train[TARGET].median()))
    rows.append({"MODEL_NAME": "TRAIN_MEDIAN_BASELINE", "MODEL_TYPE": "BASELINE", **median_val})

    fitted = {}
    for name, model in candidates().items():
        model.fit(train[FEATURES], train[TARGET])
        pred = model.predict(val[FEATURES])
        rows.append({"MODEL_NAME": name, "MODEL_TYPE": "XGBOOST" if name.startswith("XGB") else name, **metrics(val[TARGET], pred)})
        fitted[name] = model

    val_metrics = pd.DataFrame(rows)
    rule = val_metrics[val_metrics.MODEL_NAME == "DEVELOPMENT_RULE_BASELINE"].iloc[0]
    val_metrics["BEATS_RULE_MAE_FLAG"] = val_metrics.MAE < rule.MAE
    val_metrics["HIGH_STRAIN_RISK_GATE_FLAG"] = (
        val_metrics.HIGH_STRAIN_SEVERE_UNDERPREDICTION_RATE
        <= rule.HIGH_STRAIN_SEVERE_UNDERPREDICTION_RATE + 0.02
    )
    val_metrics["SELECTION_ELIGIBLE_FLAG"] = (
        val_metrics.MODEL_NAME.str.startswith("XGB")
        & val_metrics.BEATS_RULE_MAE_FLAG
        & val_metrics.HIGH_STRAIN_RISK_GATE_FLAG
    )
    val_metrics["SELECTION_SCORE"] = (
        0.45 * val_metrics.MAE + 0.20 * val_metrics.RMSE
        + 0.20 * val_metrics.HIGH_STRAIN_MAE
        + 0.15 * val_metrics.HIGH_STRAIN_SEVERE_UNDERPREDICTION_RATE
    )
    eligible = val_metrics[val_metrics.SELECTION_ELIGIBLE_FLAG].sort_values(
        ["SELECTION_SCORE", "MAE", "HIGH_STRAIN_MAE"]
    )
    if eligible.empty:
        val_metrics["SELECTED_MODEL_FLAG"] = False
        val_metrics["DATA_SPLIT"] = "VALIDATION"
        save(session, val_metrics, "TDS_VALIDATION_MODEL_METRICS_V2")
        raise RuntimeError("No XGBoost candidate passed validation safety gates. Test set was not used.")

    selected = str(eligible.iloc[0].MODEL_NAME)
    val_metrics["SELECTED_MODEL_FLAG"] = val_metrics.MODEL_NAME == selected
    val_metrics["DATA_SPLIT"] = "VALIDATION"
    val_metrics["MODEL_VERSION"] = "V2"
    val_metrics["FEATURE_SET_VERSION"] = "TDS_FEATURES_V2"
    val_metrics["DEPLOYMENT_MODE"] = "SHADOW"
    val_metrics["OFFICIAL_COST_IMPACT_ALLOWED_FLAG"] = False
    val_metrics["BUSINESS_DECISION_ALLOWED_FLAG"] = False
    val_metrics["EVALUATED_AT"] = utc_now()
    save(session, val_metrics, "TDS_VALIDATION_MODEL_METRICS_V2")

    save(session, prediction_frame(val, fitted[selected].predict(val[FEATURES]), selected, "VALIDATION"), "TDS_VALIDATION_PREDICTIONS_V2")

    train_val = pd.concat([train, val], ignore_index=True)
    final_model = candidates()[selected]
    final_model.fit(train_val[FEATURES], train_val[TARGET])
    test_pred = final_model.predict(test[FEATURES])
    test_predictions = prediction_frame(test, test_pred, selected, "TEST")
    save(session, test_predictions, "TDS_TEST_PREDICTIONS_V2")

    tm = metrics(test[TARGET], test_pred)
    rb = metrics(test[TARGET], test[RULE])
    evaluation = pd.DataFrame([{
        "MODEL_NAME": selected, "MODEL_VERSION": "V2", "DATA_SPLIT": "TEST", **tm,
        "RULE_BASELINE_MAE": rb["MAE"], "RULE_BASELINE_RMSE": rb["RMSE"],
        "RULE_HIGH_STRAIN_MAE": rb["HIGH_STRAIN_MAE"],
        "RULE_HIGH_STRAIN_UNDERPREDICTION_RATE": rb["HIGH_STRAIN_UNDERPREDICTION_RATE"],
        "MAE_IMPROVEMENT_VS_RULE": 1 - tm["MAE"] / rb["MAE"],
        "RMSE_IMPROVEMENT_VS_RULE": 1 - tm["RMSE"] / rb["RMSE"],
        "VALIDATION_STATUS": "ACCEPTED_FOR_SHADOW" if tm["MAE"] < rb["MAE"] and tm["RANGE_VIOLATION_ROWS"] == 0 else "REQUIRES_MODEL_CORRECTION",
        "DEPLOYMENT_MODE": "SHADOW",
        "OFFICIAL_COST_IMPACT_ALLOWED_FLAG": False,
        "BUSINESS_DECISION_ALLOWED_FLAG": False,
        "EVALUATED_AT": utc_now(),
    }])
    save(session, evaluation, "TDS_TEST_EVALUATION_V2")

    segments = []
    for wc, group in test_predictions.groupby("WORK_CENTER_ID"):
        segments.append({
            "MODEL_NAME": selected, "MODEL_VERSION": "V2",
            "SEGMENT_TYPE": "WORK_CENTER", "SEGMENT_VALUE": wc,
            **metrics(group[TARGET], group["PREDICTED_TDS_FACTOR"]),
            "DEPLOYMENT_MODE": "SHADOW", "EVALUATED_AT": utc_now(),
        })
    save(session, pd.DataFrame(segments), "TDS_TEST_SEGMENT_METRICS_V2")

    xgb = final_model.named_steps["model"]
    params = {k: v for k, v in xgb.get_params().items() if k in {
        "objective", "n_estimators", "learning_rate", "max_depth",
        "min_child_weight", "subsample", "colsample_bytree",
        "reg_alpha", "reg_lambda", "random_state",
    }}
    decision = pd.DataFrame([{
        "MODEL_NAME": selected, "MODEL_VERSION": "V2",
        "FEATURE_SET_VERSION": "TDS_FEATURES_V2",
        "SELECTION_DATASET": "VALIDATION",
        "SELECTION_POLICY": "BEAT_RULE_MAE;PASS_HIGH_STRAIN_RISK_GATE;MINIMIZE_WEIGHTED_SCORE",
        "MODEL_PARAMETERS_JSON": json.dumps(params, sort_keys=True),
        "TRAINING_DATA_CLASS": "SYNTHETIC", "DEPLOYMENT_MODE": "SHADOW",
        "OFFICIAL_COST_IMPACT_ALLOWED_FLAG": False,
        "BUSINESS_DECISION_ALLOWED_FLAG": False, "SELECTED_AT": utc_now(),
    }])
    save(session, decision, "TDS_SELECTED_MODEL_DECISION_V2")

    return {
        "status": "SUCCESS", "selected_model": selected,
        "validation_metrics": f"{DATABASE}.{SCHEMA}.TDS_VALIDATION_MODEL_METRICS_V2",
        "test_evaluation": f"{DATABASE}.{SCHEMA}.TDS_TEST_EVALUATION_V2",
        "test_predictions": f"{DATABASE}.{SCHEMA}.TDS_TEST_PREDICTIONS_V2",
        "deployment_mode": "SHADOW", "official_cost_impact_allowed": False,
    }

In [ ]:
session = get_active_session()
result = run(session)
result

In [ ]:
# ============================================================
# PHASE 8D.7
# Retrain the selected TDS pipeline on TRAIN + VALIDATION
# and log it into the Snowflake Model Registry.
#
# Run in a Snowflake Python Worksheet.
#
# Handler: main
# Return type: String
#
# Required worksheet packages:
#   pandas
#   scikit-learn
#   xgboost
#   snowflake-ml-python
# ============================================================

from __future__ import annotations

import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry


DATABASE = "KMAT_COST_MODEL_DB"
SCHEMA = "CORE_ML"

SOURCE_OBJECT = (
    "KMAT_COST_MODEL_DB.CORE_ML.TDS_MODEL_DATASET_V2"
)
DECISION_OBJECT = (
    "KMAT_COST_MODEL_DB.CORE_ML.TDS_SELECTED_MODEL_DECISION_V2"
)
TEST_EVALUATION_OBJECT = (
    "KMAT_COST_MODEL_DB.CORE_ML.TDS_TEST_EVALUATION_V2"
)

PHYSICAL_MODEL_NAME = "TDS_XGB_REGRESSOR"
PHYSICAL_VERSION_NAME = "V2_SHADOW_1"
EXPECTED_SELECTED_CANDIDATE = "XGB_V1_BALANCED"

TARGET_COLUMN = "TARGET_TDS_FACTOR"

CATEGORICAL_COLUMNS = [
    "KMAT_ID",
    "WORK_CENTER_ID",
    "MACHINE_ID",
    "OPERATION_ID",
    "ENGINE",
    "CAB",
    "WHEEL",
    "COLOR",
]

NUMERIC_COLUMNS = [
    "MACHINE_AGE_YEARS",
    "MACHINE_HOURS_RUN",
    "PRE_RUN_VIBRATION_MM_S",
    "PRE_RUN_TEMPERATURE_C",
    "DAYS_SINCE_MAINTENANCE",
    "TOOL_USAGE_CYCLES_PRE_RUN",
    "SETUP_TIME_MINUTES",
    "CONFIG_COMPLEXITY_SCORE",
    "STANDARD_MAINTENANCE_COST_USD",
    "SHIFT_LOAD_PCT",
    "OPERATOR_EXPERIENCE_YEARS",
    "RECENT_30D_DOWNTIME_MIN",
    "PRIOR_30D_MAINTENANCE_COUNT",
    "MACHINE_HEALTH_SCORE",
    "PRIOR_30D_MEAN_VIBRATION_MM_S",
    "PRIOR_WORK_CENTER_TDS_AVG",
]

FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS


def build_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def build_selected_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                build_one_hot_encoder(),
                CATEGORICAL_COLUMNS,
            ),
            (
                "numeric",
                "passthrough",
                NUMERIC_COLUMNS,
            ),
        ],
        remainder="drop",
    )

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=350,
        learning_rate=0.035,
        max_depth=5,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.25,
        random_state=42,
        n_jobs=1,
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


def create_run_log(session: Session) -> None:
    session.sql(
        """
        CREATE TABLE IF NOT EXISTS
            KMAT_COST_MODEL_DB.CORE_ML.TDS_MODEL_REGISTRATION_LOG_V2
        (
            RUN_ID VARCHAR,
            PHYSICAL_MODEL_NAME VARCHAR,
            PHYSICAL_VERSION_NAME VARCHAR,
            RUN_STATUS VARCHAR,
            MESSAGE VARCHAR,
            CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
        )
        """
    ).collect()


def insert_run_log(
    session: Session,
    run_id: str,
    status: str,
    message: str,
) -> None:
    safe_message = message.replace("'", "''")[:8000]

    session.sql(
        f"""
        INSERT INTO
            KMAT_COST_MODEL_DB.CORE_ML.TDS_MODEL_REGISTRATION_LOG_V2
        (
            RUN_ID,
            PHYSICAL_MODEL_NAME,
            PHYSICAL_VERSION_NAME,
            RUN_STATUS,
            MESSAGE
        )
        VALUES
        (
            '{run_id}',
            '{PHYSICAL_MODEL_NAME}',
            '{PHYSICAL_VERSION_NAME}',
            '{status}',
            '{safe_message}'
        )
        """
    ).collect()


def get_test_metrics(session: Session) -> dict:
    rows = session.table(TEST_EVALUATION_OBJECT).collect()

    if len(rows) != 1:
        raise RuntimeError(
            "TDS_TEST_EVALUATION_V2 must contain exactly one row."
        )

    row = rows[0].as_dict()

    return {
        "test_mae": float(row["MAE"]),
        "test_rmse": float(row["RMSE"]),
        "test_median_absolute_error": float(
            row["MEDIAN_ABSOLUTE_ERROR"]
        ),
        "test_high_strain_mae": float(
            row["HIGH_STRAIN_MAE"]
        ),
        "test_high_strain_underprediction_rate": float(
            row["HIGH_STRAIN_UNDERPREDICTION_RATE"]
        ),
        "test_high_strain_severe_underprediction_rate": float(
            row[
                "HIGH_STRAIN_SEVERE_UNDERPREDICTION_RATE"
            ]
        ),
        "mae_improvement_vs_rule": float(
            row["MAE_IMPROVEMENT_VS_RULE"]
        ),
        "rmse_improvement_vs_rule": float(
            row["RMSE_IMPROVEMENT_VS_RULE"]
        ),
        "deployment_mode": "SHADOW",
        "official_cost_impact_allowed": False,
        "business_decision_allowed": False,
    }


def get_or_log_model_version(
    registry: Registry,
    pipeline: Pipeline,
    sample_input: pd.DataFrame,
    metrics: dict,
):
    try:
        existing_model = registry.get_model(
            PHYSICAL_MODEL_NAME
        )
        existing_model.delete_version(
            PHYSICAL_VERSION_NAME
        )
    except Exception:
        pass

    model_version = registry.log_model(
        pipeline,
        model_name=PHYSICAL_MODEL_NAME,
        version_name=PHYSICAL_VERSION_NAME,
        comment=(
            "TDS XGB_V1_BALANCED trained on synthetic "
            "TRAIN+VALIDATION data. Shadow only. "
            "No official cost authority."
        ),
        metrics=metrics,
        sample_input_data=sample_input,
        target_platforms=["WAREHOUSE"],
        options={
            "target_methods": ["predict"],
        },
    )
    return model_version, "REGISTERED"


def main(session: Session) -> str:
    run_id = datetime.now(
        timezone.utc
    ).strftime("TDSREG_%Y%m%d_%H%M%S_%f")

    create_run_log(session)
    insert_run_log(
        session,
        run_id,
        "STARTED",
        "TDS physical model registration started.",
    )

    try:
        decision_pdf = session.table(
            DECISION_OBJECT
        ).to_pandas()
        decision_pdf.columns = [
            str(column).upper()
            for column in decision_pdf.columns
        ]

        if len(decision_pdf) != 1:
            raise RuntimeError(
                "TDS_SELECTED_MODEL_DECISION_V2 must contain "
                "exactly one row."
            )

        selected_candidate = str(
            decision_pdf.iloc[0]["MODEL_NAME"]
        )

        if selected_candidate != EXPECTED_SELECTED_CANDIDATE:
            raise RuntimeError(
                "The selected candidate does not match the "
                f"expected candidate. Found {selected_candidate}."
            )

        source_pdf = session.table(
            SOURCE_OBJECT
        ).filter(
            "DATA_SPLIT IN ('TRAIN', 'VALIDATION')"
        ).to_pandas()

        source_pdf.columns = [
            str(column).upper()
            for column in source_pdf.columns
        ]

        missing_columns = sorted(
            set(FEATURE_COLUMNS + [TARGET_COLUMN])
            .difference(source_pdf.columns)
        )

        if missing_columns:
            raise RuntimeError(
                f"Missing training columns: {missing_columns}"
            )

        if source_pdf.empty:
            raise RuntimeError(
                "TRAIN+VALIDATION dataset is empty."
            )

        if source_pdf[
            FEATURE_COLUMNS + [TARGET_COLUMN]
        ].isna().any().any():
            raise RuntimeError(
                "TRAIN+VALIDATION dataset contains null values."
            )

        X_train_validation = source_pdf[
            FEATURE_COLUMNS
        ].copy()

        y_train_validation = source_pdf[
            TARGET_COLUMN
        ].astype(float).to_numpy()

        pipeline = build_selected_pipeline()
        pipeline.fit(
            X_train_validation,
            y_train_validation,
        )

        local_smoke_prediction = pipeline.predict(
            X_train_validation.head(10)
        )

        if len(local_smoke_prediction) != 10:
            raise RuntimeError(
                "Local fitted-model smoke test returned "
                "an unexpected row count."
            )

        if not np.isfinite(local_smoke_prediction).all():
            raise RuntimeError(
                "Local fitted-model smoke test produced "
                "non-finite predictions."
            )

        metrics = get_test_metrics(session)

        registry = Registry(
            session=session,
            database_name=DATABASE,
            schema_name=SCHEMA,
        )

        model_version, action = get_or_log_model_version(
            registry=registry,
            pipeline=pipeline,
            sample_input=X_train_validation.head(100),
            metrics=metrics,
        )

        registry_smoke_result = model_version.run(
            X_train_validation.head(5),
            function_name="predict",
        )

        registry_smoke_rows = len(registry_smoke_result)

        if registry_smoke_rows != 5:
            raise RuntimeError(
                "Registered-model smoke test returned "
                f"{registry_smoke_rows} rows instead of 5."
            )

        success_message = (
            f"{action}: {PHYSICAL_MODEL_NAME}/"
            f"{PHYSICAL_VERSION_NAME}; "
            f"training_rows={len(source_pdf)}; "
            f"registry_smoke_rows={registry_smoke_rows}; "
            f"sklearn={sklearn.__version__}; "
            f"xgboost={xgboost.__version__}"
        )

        insert_run_log(
            session,
            run_id,
            "SUCCESS",
            success_message,
        )

        return json.dumps(
            {
                "status": "SUCCESS",
                "registration_action": action,
                "physical_model_name": PHYSICAL_MODEL_NAME,
                "physical_version_name": PHYSICAL_VERSION_NAME,
                "selected_candidate": selected_candidate,
                "training_rows": int(len(source_pdf)),
                "registry_smoke_rows": int(
                    registry_smoke_rows
                ),
                "deployment_mode": "SHADOW",
                "official_cost_impact_allowed": False,
                "business_decision_allowed": False,
                "sklearn_version": sklearn.__version__,
                "xgboost_version": xgboost.__version__,
            },
            sort_keys=True,
        )

    except Exception as exc:
        insert_run_log(
            session,
            run_id,
            "FAILED",
            f"{type(exc).__name__}: {str(exc)}",
        )
        raise


In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE KMAT_COST_MODEL_DB").collect()
session.sql("USE SCHEMA CORE_ML").collect()

# Drop existing model version so we can re-log with target_platforms=["WAREHOUSE"]
session.sql(
    "DROP MODEL IF EXISTS KMAT_COST_MODEL_DB.CORE_ML.TDS_XGB_REGRESSOR"
).collect()

result = main(session)
print(result)